# KRX US Core4 Backtest

This notebook backtests `krx_us_core4` from 2015 to today using the same stage helpers as the live strategy.

Assumptions:
- KRX-listed ETFs only
- Weekly rebalance on the first trading day of each week
- Z-score signal uses completed daily bars of `133690`
- Rebalance is approximated at rebalance-day close because only daily bars are used


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path

import importlib
import matplotlib.pyplot as plt
import pandas as pd

from pykis import KisAuth, PyKis
import strategies.krx_us_core4 as krx_us_core4

START_DATE = date(2015, 1, 1)
SECRET_PATH = Path('secrets/krx_us_core4.json')
CASH_RATIO = 0.03
TODAY = date.today()

krx_us_core4 = importlib.reload(krx_us_core4)
CORE4_CODES = krx_us_core4.CORE4_CODES
SIGNAL_CODE = getattr(krx_us_core4, 'SIGNAL_CODE', '133690')
build_stage_target_weights = krx_us_core4.build_stage_target_weights
compute_zscore = krx_us_core4.compute_zscore
map_zscore_to_stage = krx_us_core4.map_zscore_to_stage
resolve_applied_stage = krx_us_core4.resolve_applied_stage

kis = PyKis(KisAuth.load(SECRET_PATH), keep_token=True)


In [ ]:
def fetch_close_series(symbol: str, start: date = START_DATE, end: date = TODAY) -> pd.Series:
    chart = kis.stock(symbol).daily_chart(start=start, end=end)
    frame = pd.DataFrame(
        {
            'date': [bar.time.date() for bar in chart.bars],
            'close': [float(bar.close) for bar in chart.bars],
        }
    ).drop_duplicates(subset=['date']).sort_values('date')
    return pd.Series(frame['close'].tolist(), index=pd.Index(frame['date'].tolist(), name='date'), dtype=float)

signal_closes = fetch_close_series(SIGNAL_CODE)
asset_closes = {symbol: fetch_close_series(symbol) for symbol in CORE4_CODES}
price_frame = pd.concat(asset_closes, axis=1).sort_index().dropna()
signal_closes = signal_closes.loc[signal_closes.index.isin(price_frame.index)]
price_frame = price_frame.loc[signal_closes.index]
daily_returns = price_frame.pct_change().fillna(0.0)
signal_zscore = compute_zscore(signal_closes)

signal_frame = pd.DataFrame({'close': signal_closes, 'zscore': signal_zscore}).dropna()
aligned_zscore = signal_frame['zscore'].reindex(daily_returns.index).ffill()
signal_frame['week'] = signal_frame.index.to_series().apply(lambda d: (d.isocalendar().year, d.isocalendar().week))
rebalance_dates = signal_frame.groupby('week').apply(lambda frame: frame.index[0]).tolist()
rebalance_dates = [d for d in rebalance_dates if d in daily_returns.index]

signal_frame.tail()


In [ ]:
def run_backtest() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series, pd.Series]:
    portfolio_value = 1.0
    bucket_value = 1.0
    aggressive_value = 1.0
    fixed_value = 1.0
    previous_stage = 0
    current_weights = pd.Series(build_stage_target_weights(0))
    bucket_weights = pd.Series(build_stage_target_weights(0)) * (1 - CASH_RATIO)
    aggressive_weights = pd.Series(build_stage_target_weights(5)) * (1 - CASH_RATIO)
    fixed_weights = pd.Series(build_stage_target_weights(0)) * (1 - CASH_RATIO)
    rows = []
    portfolio_history = []
    bucket_history = []
    aggressive_history = []
    fixed_history = []

    rebalance_set = set(rebalance_dates)
    for idx, current_date in enumerate(daily_returns.index):
        if idx > 0:
            signal_date = daily_returns.index[idx - 1]
            current_zscore = float(aligned_zscore.iloc[idx - 1])
            bucket_stage = map_zscore_to_stage(current_zscore)
            applied_stage = resolve_applied_stage(previous_stage, bucket_stage)
            stage_changed = applied_stage != previous_stage
            should_rebalance = (current_date in rebalance_set) or stage_changed
            bucket_weights = pd.Series(build_stage_target_weights(bucket_stage)) * (1 - CASH_RATIO)
            if should_rebalance:
                current_weights = pd.Series(build_stage_target_weights(applied_stage)) * (1 - CASH_RATIO)
                previous_stage = applied_stage
                rows.append({
                    'rebalance_date': current_date,
                    'signal_date': signal_date,
                    'zscore': current_zscore,
                    'bucket_stage': bucket_stage,
                    'applied_stage': applied_stage,
                    'stage_changed': stage_changed,
                    '418660': current_weights['418660'],
                    '390390': current_weights['390390'],
                    '476760': current_weights['476760'],
                    '411060': current_weights['411060'],
                })

        day_returns = daily_returns.loc[current_date]
        invested_return = float((current_weights * day_returns[current_weights.index]).sum())
        bucket_return = float((bucket_weights * day_returns[bucket_weights.index]).sum())
        aggressive_return = float((aggressive_weights * day_returns[aggressive_weights.index]).sum())
        fixed_return = float((fixed_weights * day_returns[fixed_weights.index]).sum())
        portfolio_value *= 1 + invested_return
        bucket_value *= 1 + bucket_return
        aggressive_value *= 1 + aggressive_return
        fixed_value *= 1 + fixed_return
        portfolio_history.append((current_date, portfolio_value))
        bucket_history.append((current_date, bucket_value))
        aggressive_history.append((current_date, aggressive_value))
        fixed_history.append((current_date, fixed_value))

    stage_history = pd.DataFrame(rows)
    portfolio_series = pd.Series(dict(portfolio_history), name='krx_us_core4_zscore')
    bucket_series = pd.Series(dict(bucket_history), name='bucket_following')
    aggressive_series = pd.Series(dict(aggressive_history), name='always_aggressive')
    fixed_series = pd.Series(dict(fixed_history), name='krx_us_core4_fixed')
    return stage_history, portfolio_series, bucket_series, aggressive_series, fixed_series

stage_history, portfolio_series, bucket_series, aggressive_series, fixed_series = run_backtest()
stage_history.tail(), portfolio_series.tail(), bucket_series.tail(), aggressive_series.tail(), fixed_series.tail()


In [ ]:
def summarize(series: pd.Series) -> dict[str, float]:
    returns = series.pct_change().dropna()
    cumulative = float(series.iloc[-1] - 1.0)
    annualized = float((series.iloc[-1] ** (252 / max(len(returns), 1))) - 1) if len(returns) else 0.0
    drawdown = series / series.cummax() - 1
    mdd = float(drawdown.min()) if len(drawdown) else 0.0
    volatility = float(returns.std() * (252 ** 0.5)) if len(returns) else 0.0
    return {
        'cumulative_return': cumulative,
        'annualized_return': annualized,
        'max_drawdown': mdd,
        'annualized_volatility': volatility,
    }

summary = pd.DataFrame({
    'krx_us_core4_zscore': summarize(portfolio_series),
    'bucket_following': summarize(bucket_series),
    'always_aggressive': summarize(aggressive_series),
    'krx_us_core4_fixed': summarize(fixed_series),
}).T
summary


In [ ]:
signal_plot = signal_frame.copy()
signal_plot.index = pd.to_datetime(signal_plot.index)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
signal_plot['close'].plot(ax=ax1, color='tab:blue', title='133690 Price and Z-score')
ax1.set_ylabel('Close')

signal_plot['zscore'].plot(ax=ax2, color='tab:orange', label='Z-score')
for level in [0, -1.0, -2.0, -3.0, -4.0]:
    ax2.axhline(level, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.set_ylabel('Z-score')
ax2.text(signal_plot.index[0], 0.3, 'Stage buckets: 0 / -1 / -2 / -3 / -4', fontsize=9, color='gray')
ax2.legend(loc='upper left')
plt.show()


In [ ]:
plot_frame = pd.concat([portfolio_series, bucket_series, aggressive_series, fixed_series], axis=1)
stage_plot = stage_history[['rebalance_date', 'applied_stage']].drop_duplicates(subset=['rebalance_date']).copy()
stage_plot['rebalance_date'] = pd.to_datetime(stage_plot['rebalance_date'])
stage_plot = stage_plot.set_index('rebalance_date').reindex(pd.to_datetime(plot_frame.index)).ffill().fillna(0)

fig, ax1 = plt.subplots(figsize=(14, 6))
plot_frame.plot(ax=ax1, title='KRX US Core4 Backtest with Applied Stage')
ax1.set_ylabel('Cumulative NAV')

ax2 = ax1.twinx()
ax2.step(plot_frame.index, stage_plot['applied_stage'], where='post', color='black', alpha=0.35, label='applied_stage')
ax2.set_ylabel('Applied Stage')
ax2.set_ylim(-0.2, 5.2)
ax2.set_yticks(range(6))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.show()

print('Note: higher applied_stage means more aggressive attack exposure.')
stage_history[['rebalance_date', 'signal_date', 'zscore', 'bucket_stage', 'applied_stage']].tail(20)
